* This example demonstrates how to use machine-learning models in alpha-mind.

* The original example asked users to set `DB_URI` to a database. This version uses deterministic local data, so neither `DB_URI` nor a database is required.

In [ ]:
import asyncio


## Data Configuration
------------

In [ ]:
import datetime as dt
import numpy as np


I use the current values of `["EMA5D", "EMV6D"]` to predict returns approximately one month ahead.

* The training universe is `hs300`.
* All factors are neutralized, standardized, and otherwise preprocessed.
* For the linear model, observations are spaced ten business days apart, and the previous eight periods provide the training features.

In [ ]:
import pyarrow as pa


## Model Prediction
-----------

In [ ]:
import pyarrow.compute as pc

rng = np.random.default_rng(42)


In [ ]:
n_dates = 6


## Model Comparison (Linear Regression vs. Naive Constant Linear Model)
------------------

In [ ]:
n_assets = 12


In [ ]:
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])


In [ ]:
n_rows = n_dates * n_assets


## Model Time Series
-------------------------

By comparing test-set results, I can evaluate the following models:

* Naive Model: uses the previous factor value directly as the current prediction.
* Linear regression: fits previous factor values and uses the resulting model to predict the current value.

In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
from alphamind.model import LinearRegression, load_model


In [ ]:
train = panel.filter(pc.less(panel["trade_date"], dates[-1]))


In this example, the linear regression model has a slightly higher IC than the Naive model.

## Backtest (Simple Long-Short Strategy)
--------------------------

In [ ]:
test = panel.filter(pc.equal(panel["trade_date"], dates[-1]))


In [ ]:
model = LinearRegression(features=["factor", "value"], fit_intercept=True)
model.fit(train, train["dx"].to_numpy(zero_copy_only=False))


In [ ]:
    ## naive model
    ## regression model
predictions = pa.table({
    "trade_date": test["trade_date"],
    "code": test["code"],
    "prediction": model.predict(test),
})


In [ ]:
restored = load_model(model.save())
np.testing.assert_allclose(restored.predict(test), predictions["prediction"].to_pylist())


In [ ]:
predictions
